# Molecular Docking of Natural Compounds Against Human Acetylcholinesterase (PDB: 4EY7)

## Objective

To identify potential natural acetylcholinesterase inhibitors using molecular docking with AutoDock Vina.

---

## Workflow

1. Install required software
2. Upload receptor
3. Upload ligands
4. Prepare receptor
5. Prepare ligands
6. Perform docking
7. Rank compounds
8. Visualize best complexes

Mount Google Drive

In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
import os

project_dir = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening"

protein_file = os.path.join(project_dir, "Protein", "4EY7_clean.pdb")
ligand_dir = os.path.join(project_dir, "Ligands", "SDF")

print("Protein exists:", os.path.exists(protein_file))

ligands = [f for f in os.listdir(ligand_dir) if f.endswith(".sdf")]

print("Number of ligands:", len(ligands))

print("\nFirst 10 ligands:")
for ligand in sorted(ligands)[:10]:
    print(ligand)

Protein exists: True
Number of ligands: 41

First 10 ligands:
Apigenin.sdf
Asiatic acid.sdf
Baicalein.sdf
Berberine.sdf
Bilobalide.sdf
Bisdemethoxycurcumin.sdf
Caffeic acid.sdf
Catechin.sdf
Chlorogenic acid.sdf
Chrysin.sdf


Install the required packages

In [40]:
!pip -q install rdkit meeko vina pandas tqdm biopython

Test the installation

In [41]:
import rdkit
import vina
import pandas

print("RDKit installed ✓")
print("Vina installed ✓")
print("Pandas installed ✓")

RDKit installed ✓
Vina installed ✓
Pandas installed ✓


In [42]:
from Bio.PDB import PDBParser
print("Biopython installed successfully!")

Biopython installed successfully!


Find the ligand name

In [43]:
from Bio.PDB import PDBParser

pdb_path = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Protein/4EY7.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("4EY7", pdb_path)

ligands = set()

for model in structure:
    for chain in model:
        for residue in chain:
            hetflag = residue.id[0]
            if hetflag.startswith("H_"):
                ligands.add(residue.resname)

print("Ligands found:")
print(sorted(ligands))

Ligands found:
['E20', 'EDO', 'FUC', 'NAG', 'NO3']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Load the PDB

In [44]:
from Bio.PDB import PDBParser

pdb_path = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Protein/4EY7.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("4EY7", pdb_path)

print("Protein loaded successfully!")

Protein loaded successfully!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Find all hetero residues

In [45]:
hetero_residues = set()

for model in structure:
    for chain in model:
        for residue in chain:
            if residue.id[0] != " ":
                hetero_residues.add(residue.resname)

print("Hetero residues found:")
print(sorted(hetero_residues))

Hetero residues found:
['E20', 'EDO', 'FUC', 'HOH', 'NAG', 'NO3']


Calculate the Center of Donepezil (E20)

In [46]:
import numpy as np

coords = []

for model in structure:
    for chain in model:
        for residue in chain:
            if residue.resname == "E20":
                for atom in residue:
                    coords.append(atom.coord)

coords = np.array(coords)

center = coords.mean(axis=0)

print("Center coordinates:")
print(f"X = {center[0]:.3f}")
print(f"Y = {center[1]:.3f}")
print(f"Z = {center[2]:.3f}")

Center coordinates:
X = -1.581
Y = -50.192
Z = 2.101


In [47]:
import numpy as np

coords = []

for model in structure:
    for chain in model:
        for residue in chain:
            if residue.resname == "E20":
                for atom in residue:
                    coords.append(atom.coord)

coords = np.array(coords)

mins = coords.min(axis=0)
maxs = coords.max(axis=0)

print("Minimum coordinates:")
print(mins)

print("\nMaximum coordinates:")
print(maxs)

print("\nLigand dimensions (Å):")
print(maxs - mins)

Minimum coordinates:
[-20.658 -63.54  -26.602]

Maximum coordinates:
[ 13.999 -41.266  32.286]

Ligand dimensions (Å):
[34.657    22.274002 58.888   ]


where E20 is located

In [48]:
for model in structure:
    for chain in model:
        for residue in chain:
            if residue.resname == "E20":
                print(
                    f"Chain: {chain.id}, Residue ID: {residue.id[1]}, Atoms: {len(list(residue.get_atoms()))}"
                )

Chain: A, Residue ID: 604, Atoms: 28
Chain: B, Residue ID: 605, Atoms: 28


Calculate the correct docking center (Chain A only)

In [49]:
import numpy as np

coords = []

for model in structure:
    for chain in model:
        if chain.id == "A":
            for residue in chain:
                if residue.resname == "E20":
                    for atom in residue:
                        coords.append(atom.coord)

coords = np.array(coords)

center = coords.mean(axis=0)

print("Docking Center")
print("----------------")
print(f"Center X = {center[0]:.3f}")
print(f"Center Y = {center[1]:.3f}")
print(f"Center Z = {center[2]:.3f}")

mins = coords.min(axis=0)
maxs = coords.max(axis=0)

print("\nLigand Size")
print("----------------")
print(maxs - mins)

Docking Center
----------------
Center X = -14.108
Center Y = -43.833
Center Z = 27.670

Ligand Size
----------------
[13.3410015  5.2789993 10.355    ]


Install receptor preparation tools

In [50]:
!pip -q install meeko gemmi

In [51]:
import meeko
import gemmi

print("Meeko:", meeko.__version__)
print("Gemmi installed successfully!")

Meeko: 0.7.1
Gemmi installed successfully!


Inspect the cleaned protein

In [52]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure(
    "4EY7",
    "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Protein/4EY7_clean.pdb"
)

for model in structure:
    for chain in model:
        for residue in chain:
            if chain.id == "A" and residue.id[1] in [54, 313, 381, 525]:
                print(f"\nResidue {chain.id}:{residue.id[1]} {residue.resname}")
                for atom in residue:
                    if atom.is_disordered():
                        print(atom.name, "AltLoc:", atom.get_altloc())


Residue A:54 GLN
N AltLoc: B
CA AltLoc: B
C AltLoc: B
O AltLoc: B
CB AltLoc: B
CG AltLoc: B
CD AltLoc: B
OE1 AltLoc: B
NE2 AltLoc: B

Residue A:313 GLU
N AltLoc: B
CA AltLoc: B
C AltLoc: B
O AltLoc: B
CB AltLoc: B
CG AltLoc: B
CD AltLoc: B
OE1 AltLoc: B
OE2 AltLoc: B

Residue A:381 HIS
N AltLoc: A
CA AltLoc: A
C AltLoc: A
O AltLoc: A
CB AltLoc: A
CG AltLoc: A
ND1 AltLoc: A
CD2 AltLoc: A
CE1 AltLoc: A
NE2 AltLoc: A

Residue A:525 ARG
N AltLoc: B
CA AltLoc: B
C AltLoc: B
O AltLoc: B
CB AltLoc: B
CG AltLoc: B
CD AltLoc: B
NE AltLoc: B
CZ AltLoc: B
NH1 AltLoc: B
NH2 AltLoc: B


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Receptor Preparation

In [53]:
import os

project_dir = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening"

protein_file = os.path.join(project_dir, "Protein", "4EY7_clean.pdb")

print("Protein file exists:", os.path.exists(protein_file))
print("Protein file:", protein_file)

Protein file exists: True
Protein file: /content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Protein/4EY7_clean.pdb


In [54]:
import meeko

print("Meeko version:", meeko.__version__)

Meeko version: 0.7.1


In [55]:
!which mk_prepare_receptor.py

/usr/local/bin/mk_prepare_receptor.py


In [56]:
!which mk_prepare_ligand.py

/usr/local/bin/mk_prepare_ligand.py


Create an output folder

In [57]:
import os

project_dir = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening"

protein_dir = os.path.join(project_dir, "Protein")
prepared_dir = os.path.join(project_dir, "Prepared_Protein")

os.makedirs(prepared_dir, exist_ok=True)

print(prepared_dir)

/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Protein


Install ProDy

In [58]:
!pip -q install prody

In [59]:
!mk_prepare_receptor.py \
--read_pdb "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Protein/4EY7_clean.pdb" \
--wanted_altloc "A:54=B,A:313=B,A:381=A,A:525=B" \
-o "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Protein/4EY7" \
-p


Files written:
/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Protein/4EY7.pdbqt <-- static (i.e., rigid) receptor input file


In [60]:
import os

prepared_dir = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Protein"
print(os.listdir(prepared_dir))

['4EY7.pdbqt']


Extract the Crystal Ligand (Donepezil)

Extract E20 from 4EY7.pdb

In [61]:
from Bio.PDB import PDBParser, PDBIO, Select
import os

project_dir = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening"

input_pdb = os.path.join(project_dir, "Protein", "4EY7.pdb")
output_pdb = os.path.join(project_dir, "Protein", "donepezil.pdb")

parser = PDBParser(QUIET=True)
structure = parser.get_structure("4EY7", input_pdb)

class DonepezilSelect(Select):
    def accept_residue(self, residue):
        return (
            residue.resname == "E20"
            and residue.get_parent().id == "A"
        )

io = PDBIO()
io.set_structure(structure)
io.save(output_pdb, DonepezilSelect())

print("Donepezil extracted successfully!")
print(output_pdb)

Donepezil extracted successfully!
/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Protein/donepezil.pdb


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Verify the ligand

In [62]:
import os

ligand_file = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Protein/donepezil.pdb"

print("Exists:", os.path.exists(ligand_file))
print("Size:", os.path.getsize(ligand_file), "bytes")

Exists: True
Size: 2357 bytes


Inspect the file

In [63]:
with open(ligand_file) as f:
    for i in range(10):
        print(f.readline().rstrip())

HETATM    1  C1  E20 A 604     -18.530 -41.928  24.258  1.00 28.11           C
HETATM    2  C2  E20 A 604     -18.970 -41.862  25.566  1.00 29.82           C
HETATM    3  C3  E20 A 604     -18.125 -42.222  26.615  1.00 27.24           C
HETATM    4  C4  E20 A 604     -16.817 -42.659  26.338  1.00 28.88           C
HETATM    5  C5  E20 A 604     -16.370 -42.740  25.037  1.00 21.11           C
HETATM    6  C6  E20 A 604     -17.234 -42.369  23.979  1.00 24.23           C
HETATM    7  C7  E20 A 604     -14.962 -43.221  24.996  1.00 21.45           C
HETATM    8  C8  E20 A 604     -14.502 -43.478  26.419  1.00 21.95           C
HETATM    9  C9  E20 A 604     -15.710 -43.107  27.282  1.00 23.11           C
HETATM   10  C10 E20 A 604     -13.970 -44.894  26.642  1.00 23.90           C


Prepare ALL Ligands Automatically

Create the output folder

In [64]:
import os

project_dir = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening"

ligand_input = os.path.join(project_dir, "Ligands", "SDF")
ligand_output = os.path.join(project_dir, "Prepared_Ligands")

os.makedirs(ligand_output, exist_ok=True)

print("Input :", ligand_input)
print("Output:", ligand_output)

Input : /content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Ligands/SDF
Output: /content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Ligands


Check the number of ligands

In [65]:
import os

ligands = sorted([f for f in os.listdir(ligand_input) if f.endswith(".sdf")])

print("Total ligands:", len(ligands))
print("\nFirst 10 ligands:")

for ligand in ligands[:10]:
    print(ligand)

Total ligands: 41

First 10 ligands:
Apigenin.sdf
Asiatic acid.sdf
Baicalein.sdf
Berberine.sdf
Bilobalide.sdf
Bisdemethoxycurcumin.sdf
Caffeic acid.sdf
Catechin.sdf
Chlorogenic acid.sdf
Chrysin.sdf


Prepare one ligand first (Test)

In [70]:
!mk_prepare_ligand.py \
-i "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Ligands/SDF/Donepezil.sdf" \
-o "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Ligands/Donepezil.pdbqt"

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


Verify the output

In [73]:
import os

print(os.listdir("/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Ligands"))

['Donepezil.pdbqt']


First Docking

Verify AutoDock Vina

In [74]:
from vina import Vina

print("AutoDock Vina imported successfully!")

AutoDock Vina imported successfully!


Create the Vina object

In [75]:
from vina import Vina

v = Vina(sf_name='vina')

Load the receptor

In [76]:
v.set_receptor(
    "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Protein/4EY7.pdbqt"
)

print("Receptor loaded!")

Receptor loaded!


Generate the affinity maps

In [77]:
center = [-14.108, -43.833, 27.670]
box_size = [24, 24, 24]

v.compute_vina_maps(center=center, box_size=box_size)

print("Maps generated!")

Maps generated!


In [81]:
import os

ligand = "/content/drive/MyDrive/Alzheimer_AI_Virtual_Screening/Prepared_Ligands/Donepezil.pdbqt"

print("Exists:", os.path.exists(ligand))
print("Size:", os.path.getsize(ligand))

Exists: True
Size: 2745


In [82]:
with open(ligand) as f:
    for i in range(20):
        print(f.readline().rstrip())

REMARK SMILES COc1cc2c(cc1OC)C(=O)[C@H](CC1CCN(Cc3ccccc3)CC1)C2
REMARK SMILES IDX 14 1 13 2 12 3 11 4 6 5 28 6 5 7 7 8 4 9 8 10 3 11 9 12
REMARK SMILES IDX 10 13 2 14 1 15 15 16 16 17 27 18 17 19 26 20 18 21 19 22
REMARK SMILES IDX 20 23 21 24 25 25 22 26 24 27 23 28
REMARK H PARENT
ROOT
ATOM      1  C   UNL     1       0.179  -1.613  -0.153  1.00  0.00     0.016 C
ENDROOT
BRANCH   1   2
ATOM      2  C   UNL     1       1.045  -0.570   0.527  1.00  0.00     0.066 C
ATOM      3  O   UNL     1       2.281  -2.129   1.954  1.00  0.00    -0.294 OA
ATOM      4  C   UNL     1       2.270  -1.202   1.164  1.00  0.00     0.166 C
ATOM      5  C   UNL     1       3.466  -0.521   0.694  1.00  0.00     0.025 A
ATOM      6  C   UNL     1       1.607   0.474  -0.465  1.00  0.00     0.048 C
ATOM      7  C   UNL     1       3.084   0.444  -0.229  1.00  0.00    -0.033 A
ATOM      8  C   UNL     1       4.782  -0.739   1.056  1.00  0.00     0.062 A
ATOM      9  C   UNL     1       4.048   1.238  -0.831 